In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import random # Moved up to be available for initial copying

#Clean corrupted / non-image files - moving valid_extensions definition up
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

# Copy datasets to Colab VM
!mkdir -p /content/data/real
!mkdir -p /content/data/fake

real_source_dir = os.path.join(path, "RealVsFake", "RealVsFake", "Real")
fake_source_dir = os.path.join(path, "RealVsFake", "RealVsFake", "Fake")

# Define initial subset size for copying to /content/data/real and /content/data/fake
# This prevents the "Argument list too long" error for large datasets
# and provides the base pool of images for the split_data function.
# Let's copy slightly more than the 'subset_size' used in split_data to ensure enough images.
initial_copy_pool_size = 2500 # Ensure at least 2000 are copied for split_data

# --- Copy a subset of Real images ---
print(f"Copying a subset of real images from {real_source_dir} to /content/data/real/")
all_real_files = [f for f in os.listdir(real_source_dir) if f.lower().endswith(valid_extensions)]
random.shuffle(all_real_files)
files_to_copy_real = all_real_files[:min(initial_copy_pool_size, len(all_real_files))]

for f in files_to_copy_real:
    shutil.copy2(os.path.join(real_source_dir, f), "/content/data/real/")
print(f"Finished copying {len(files_to_copy_real)} real images.")

# --- Copy a subset of Fake images ---
print(f"Copying a subset of fake images from {fake_source_dir} to /content/data/fake/")
all_fake_files = [f for f in os.listdir(fake_source_dir) if f.lower().endswith(valid_extensions)]
random.shuffle(all_fake_files)
files_to_copy_fake = all_fake_files[:min(initial_copy_pool_size, len(all_fake_files))]

for f in files_to_copy_fake:
    shutil.copy2(os.path.join(fake_source_dir, f), "/content/data/fake/")
print(f"Finished copying {len(files_to_copy_fake)} fake images.")

# The rest of the cell from here is mostly fine, as split_data will now work with populated directories.

def clean_folder(folder):
    removed = 0
    for root, dirs, files in os.walk(folder):
        for f in files:
            path = os.path.join(root, f)
            if not f.lower().endswith(valid_extensions):
                os.remove(path)
                removed += 1
                continue
            try:
                img = Image.open(path)
                img.verify()
            except Exception as e: # Catch specific exception for more robust error handling
                print(f"Error processing {path}: {e}")
                os.remove(path)
                removed += 1
    return removed

print("Cleaning real images...")
removed_real = clean_folder("/content/data/real")
print(f"Removed {removed_real} invalid real images ")

print("Cleaning fake images...")
removed_fake = clean_folder("/content/data/fake")
print(f"Removed {removed_fake} invalid fake images")

#  Split into train/validation folders

def split_data(source_folder, train_folder, val_folder, split_ratio=0.8, subset_size=None):
    os.makedirs(train_folder, exist_ok=True)
    os.makedirs(val_folder, exist_ok=True)
    files = [f for f in os.listdir(source_folder) if f.lower().endswith(valid_extensions)]

    if subset_size:
        files = random.sample(files, min(subset_size, len(files)))  # take subset for faster training

    random.shuffle(files)
    split = int(len(files) * split_ratio)

    for f in files[:split]:
        shutil.copy(os.path.join(source_folder, f), os.path.join(train_folder, f))
    for f in files[split:]:
        shutil.copy(os.path.join(source_folder, f), os.path.join(val_folder, f))

    print(f"Split {len(files)} images → {len(files[:split])} train, {len(files[split:])} val")

# Folders for train/val
train_real = "/content/data/train/real"
val_real   = "/content/data/val/real"
train_fake = "/content/data/train/fake"
val_fake   = "/content/data/val/fake"

# Create train/val split
split_data("/content/data/real", train_real, val_real, subset_size=2000)   # use 2000 real images
split_data("/content/data/fake", train_fake, val_fake, subset_size=2000)   # use 2000 fake images

#  ImageDataGenerator
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.1
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    "/content/data/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    "/content/data/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

print("Class mapping:", train_generator.class_indices)
# Expected: {'fake': 0, 'real': 1}

#  Load MobileNetV2 pre-trained model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # freeze base model

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  # binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

#  Train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Path to your test image
test_image_path = "/content/cyberpunk-illustration-with-neon-colors-futuristic-technology.jpg"

# Load the image with same target size as training
img = image.load_img(test_image_path, target_size=(IMG_SIZE, IMG_SIZE))

# Convert to array and scale
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)  # add batch dimension

# Predict
prediction = model.predict(img_array)[0][0]

# Interpret prediction
if prediction >= 0.5:
    print(f"Prediction: Real face(confidence: {prediction:.2f})")
else:
    print(f"Prediction: Fake/AI-generated face  (confidence: {1-prediction:.2f})")